In [1]:
%pip install -q faiss-gpu

In [2]:
import faiss
import pickle
import os

In [3]:

from huggingface_hub import snapshot_download
import os
data_dir = snapshot_download(repo_id='l3mon3/embedded_vector', repo_type='dataset', local_dir='/content/embedded')
EMBEDDED_PATH = '/content/embedded'
index = faiss.read_index(os.path.join(EMBEDDED_PATH, "faiss.index"))
with open(os.path.join(EMBEDDED_PATH, "meata.pkl"), "rb") as f:
    meta = pickle.load(f)
corpus_ids_full = meta["ids"]
corpus_texts_full = meta["texts"]

print("Số vector trong index:", index.ntotal)
print("Số ids:", len(corpus_ids_full))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Số vector trong index: 224008
Số ids: 224008


In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer
import torch

model = SentenceTransformer('BAAI/bge-m3', device = 'cuda')
model.max_seq_length = 1024
model.half()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({})
)

In [5]:
def search(query: str, top_k: int = 5):
    # Encode câu hỏi thành vector, dùng ĐÚNG model đã encode corpus (BGE-M3)
    q_emb = model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")

    # Tìm top_k vector gần nhất trong index
    scores, indices = index.search(q_emb, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "id": corpus_ids_full[idx],
            "text": corpus_texts_full[idx],
            "score": float(score),
        })
    return results

In [6]:
query = "Điều kiện để ly hôn đơn phương là gì?"
results = search(query, top_k=5)

for i, r in enumerate(results, 1):
    print(f"[{i}] score={r['score']:.4f} | id={r['id']}")
    print(r['text'][:300])
    print("---")

[1] score=0.6628 | id=78761
Căn cứ cho ly hôn
1. Tòa án công nhận thuận tình ly hôn đối với trường hợp vợ chồng cùng yêu cầu ly hôn khi đáp ứng đủ các điều kiện sau:
a) Hai bên tự nguyện ly hôn;
b) Đã thỏa thuận về việc chia tài sản, việc trông nom, nuôi dưỡng, chăm sóc, giáo dục con trên cơ sở bảo đảm quyền lợi chính đáng của
---
[2] score=0.6563 | id=47617
"Điều 55. Thuận tình ly hôn
Trong trường hợp vợ chồng cùng yêu cầu ly hôn, nếu xét thấy hai bên thật sự tự nguyện ly hôn và đã thỏa thuận về việc chia tài sản, việc trông nom, nuôi dưỡng, chăm sóc, giáo dục con trên cơ sở bảo đảm quyền lợi chính đáng của vợ và con thì Tòa án công nhận thuận tình ly 
---
[3] score=0.6494 | id=336000
Mục 1. LY HÔN
Điều 51. Quyền yêu cầu giải quyết ly hôn
1. Vợ, chồng hoặc cả hai người có quyền yêu cầu Tòa án giải quyết ly hôn.
2. Cha, mẹ, người thân thích khác có quyền yêu cầu Tòa án giải quyết ly hôn khi một bên vợ, chồng do bị bệnh tâm thần hoặc mắc bệnh khác mà không thể nhận thức, làm chủ đư
---


In [7]:
import torch
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM

LLM_NAME = 'Qwen/Qwen3-4B-Instruct-2507'
# Quantization config
nf4_config = BitsAndBytesConfig(load_in_4bit=True,
    bnb_4bit_quant_type = 'nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

In [8]:
%pip install -U -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 4.7 MB/s eta 0:00:00


In [9]:
llm_model = AutoModelForCausalLM.from_pretrained(LLM_NAME,
                                             quantization_config=nf4_config,
                                             low_cpu_mem_usage=True)

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

In [10]:
def build_prompt(query: str, passages: list[dict]) -> str:
    context_block = "\n\n".join(
        f"[{i+1}] {p['text']}" for i, p in enumerate(passages)
    )
    return (
        f"NGỮ CẢNH PHÁP LUẬT:\n{context_block}\n\n"
        f"CÂU HỎI: {query}\n\n"
        "Hãy trả lời câu hỏi trên bằng tiếng Việt, CHỈ dựa vào ngữ cảnh đã cho. "
        "Nếu ngữ cảnh không đủ thông tin, hãy nói rõ là không tìm thấy thông tin phù hợp, không tự bịa."
    )


def generate_answer(query: str, top_k: int = 5, max_new_tokens: int = 512) -> str:
    passages = search(query, top_k=top_k)
    prompt = build_prompt(query, passages)

    messages = [{"role": "user", "content": prompt}]
    text = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = llm_tokenizer([text], return_tensors="pt").to(llm_model.device)
    output_ids = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.3,
    )

    new_tokens = output_ids[0][inputs.input_ids.shape[1]:]
    answer = llm_tokenizer.decode(new_tokens, skip_special_tokens=True)
    return answer, passages

In [11]:
query = "Điều kiện để ly hôn đơn phương là gì?"
answer, sources = generate_answer(query, top_k=5)

print("CÂU HỎI:", query)
print("\nCÂU TRẢ LỜI:")
print(answer)

print("\nNGUỒN THAM KHẢO:")
for i, s in enumerate(sources, 1):
    print(f"[{i}] (score={s['score']:.4f}) {s['text'][:150]}...")

CÂU HỎI: Điều kiện để ly hôn đơn phương là gì?

CÂU TRẢ LỜI:
Điều kiện để ly hôn đơn phương được nêu rõ tại các điều khoản trong ngữ cảnh đã cho, cụ thể như sau:

1. **Khi vợ hoặc chồng yêu cầu ly hôn mà hòa giải tại Tòa án không thành thì Tòa án giải quyết cho ly hôn nếu có căn cứ về việc vợ, chồng có hành vi bạo lực gia đình hoặc vi phạm nghiêm trọng quyền, nghĩa vụ của vợ, chồng làm cho hôn nhân lâm vào tình trạng trầm trọng, đời sống chung không thể kéo dài, mục đích của hôn nhân không đạt được.**

2. **Trong trường hợp vợ hoặc chồng của người bị Tòa án tuyên bố mất tích yêu cầu ly hôn thì Tòa án giải quyết cho ly hôn.**

3. **Trong trường hợp có yêu cầu ly hôn theo quy định tại khoản 2 Điều 51 của Luật này thì Tòa án giải quyết cho ly hôn nếu có căn cứ về việc chồng, vợ có hành vi bạo lực gia đình làm ảnh hưởng nghiêm trọng đến tính mạng, sức khỏe, tinh thần của người kia.**

Tuy nhiên, cần lưu ý rằng **theo Điều 51, khoản 3**, **chồng không có quyền yêu cầu ly hôn trong trường hợ